### Build Kaggle submission


Build the submission file from separate men's and women's models.


In [78]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

In [79]:
data_dir = Path("../data")
processed_dir = Path("../data/processed")
artifacts_dir = Path("../artifacts")
output_dir = Path("../submissions")
output_dir.mkdir(parents=True, exist_ok=True)

### Submission stage

In [80]:
submission_stage = "auto"

### Helper functions

In [81]:
def parse_submission_ids(submission_df):
    parsed = submission_df.copy()
    parsed[["Season", "Team1ID", "Team2ID"]] = (
        parsed["ID"].str.split("_", expand=True).astype(int)
    )
    return parsed

In [82]:
def resolve_submission_file(stage="auto"):
    stage_files = {
        "stage1": data_dir / "SampleSubmissionStage1.csv",
        "stage2": data_dir / "SampleSubmissionStage2.csv",
    }

    if stage in stage_files:
        if not stage_files[stage].exists():
            raise FileNotFoundError(f"{stage_files[stage]} does not exist.")
        return stage, stage_files[stage]

    latest_stage = None
    latest_season = -1

    for stage_name, path in stage_files.items():
        if not path.exists():
            continue

        sample_df = pd.read_csv(path, usecols=["ID"])
        seasons = sample_df["ID"].str.split("_", expand=True)[0].astype(int)
        stage_max_season = int(seasons.max())

        if stage_max_season > latest_season:
            latest_stage = stage_name
            latest_season = stage_max_season

    if latest_stage is None:
        raise FileNotFoundError("No sample submission file found in ../data.")

    return latest_stage, stage_files[latest_stage]

In [83]:
def add_gender(sample_df, men_team_ids, women_team_ids):
    sample_df = parse_submission_ids(sample_df)

    men_mask = (
            sample_df["Team1ID"].isin(men_team_ids)
            & sample_df["Team2ID"].isin(men_team_ids)
    )

    women_mask = (
            sample_df["Team1ID"].isin(women_team_ids)
            & sample_df["Team2ID"].isin(women_team_ids)
    )

    sample_df["Gender"] = np.where(
        men_mask,
        "M",
        np.where(women_mask, "W", "UNKNOWN")
    )

    return sample_df

In [84]:
def merge_team_features(df, team_features):
    df = df.copy()

    team1_features = team_features.add_prefix("Team1_")
    team2_features = team_features.add_prefix("Team2_")

    merged = df.merge(
        team1_features,
        left_on=["Season", "Team1ID"],
        right_on=["Team1_Season", "Team1_TeamID"],
        how="left"
    )

    merged = merged.merge(
        team2_features,
        left_on=["Season", "Team2ID"],
        right_on=["Team2_Season", "Team2_TeamID"],
        how="left"
    )

    return merged

In [85]:
def add_submission_features(df, ranking_col):
    df = df.copy()

    def safe_diff(col1, col2, default=np.nan):
        if col1 in df.columns and col2 in df.columns:
            return df[col1] - df[col2]
        return pd.Series(default, index=df.index)

    df["SeedNumDiff"] = safe_diff("Team1_SeedNum", "Team2_SeedNum")
    df["RankingDiff"] = safe_diff(f"Team1_{ranking_col}", f"Team2_{ranking_col}")
    df["MarginDiff"] = safe_diff("Team1_AvgMarginScore", "Team2_AvgMarginScore")
    df["NetRatingDiff"] = safe_diff("Team1_AvgNetRating", "Team2_AvgNetRating")
    df["OffEffDiff"] = safe_diff("Team1_AvgOffEfficiency", "Team2_AvgOffEfficiency")
    df["DefEffDiff"] = safe_diff("Team1_AvgDefEfficiency", "Team2_AvgDefEfficiency")
    df["WinPctDiff"] = safe_diff("Team1_WinPct", "Team2_WinPct")
    df["TurnoverMarginDiff"] = safe_diff("Team1_AvgTurnoverMargin", "Team2_AvgTurnoverMargin")
    df["ReboundPctDiff"] = safe_diff("Team1_AvgReboundPct", "Team2_AvgReboundPct")

    df["OffDefGap"] = df["OffEffDiff"] - df["DefEffDiff"]
    df["DominanceScore"] = df["MarginDiff"] * df["WinPctDiff"]
    df["NetRating_Margin_Interaction"] = df["NetRatingDiff"] * df["MarginDiff"]
    df["Margin_Ranking_Interaction"] = df["MarginDiff"] * df["RankingDiff"]

    df.replace([np.inf, -np.inf], 0, inplace=True)

    return df

In [86]:
def ensure_model_features(df, feature_list, fill_values, dataset_name="dataset"):
    df = df.copy()

    missing_cols = [col for col in feature_list if col not in df.columns]
    if missing_cols:
        print(f"\n[{dataset_name}] Adding missing model columns:")
        print(missing_cols)
        for col in missing_cols:
            df[col] = fill_values.get(col, 0.0)

    df[feature_list] = df[feature_list].replace([np.inf, -np.inf], np.nan)
    df[feature_list] = df[feature_list].fillna(fill_values)

    return df

In [87]:
def print_missing_summary(df, feature_list, label):
    missing = df[feature_list].isna().sum().sort_values(ascending=False)
    print(f"\n{label} missing values:")
    print(missing[missing > 0] if (missing > 0).any() else "None")


### Resolve which sample submission file to use

In [88]:
selected_stage, submission_path = resolve_submission_file(submission_stage)
sample_submission = pd.read_csv(submission_path)

print(f"Using {selected_stage}: {submission_path.name}")
print(f"Submission rows: {len(sample_submission)}")

Using stage2: SampleSubmissionStage2.csv
Submission rows: 132133


### Load saved models and saved feature lists

In [89]:
m_model = joblib.load(artifacts_dir / "best_model_men.pkl")
w_model = joblib.load(artifacts_dir / "best_model_women.pkl")

In [90]:
with open(artifacts_dir / "best_model_men_features.json", "r") as f:
    m_features = json.load(f)

with open(artifacts_dir / "best_model_women_features.json", "r") as f:
    w_features = json.load(f)

In [91]:
print("Men features:")
print(m_features)
print("\nWomen features:")
print(w_features)

Men features:
['SeedNumDiff', 'RankingDiff', 'MarginDiff', 'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'WinPctDiff', 'OffDefGap', 'DominanceScore', 'NetRating_Margin_Interaction', 'Margin_Ranking_Interaction', 'TurnoverMarginDiff', 'ReboundPctDiff']

Women features:
['SeedNumDiff', 'RankingDiff', 'MarginDiff', 'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'WinPctDiff', 'OffDefGap', 'DominanceScore', 'NetRating_Margin_Interaction', 'Margin_Ranking_Interaction', 'TurnoverMarginDiff', 'ReboundPctDiff']


### Load processed training datasets only to compute fill values

In [92]:
m_train_df = pd.read_csv(processed_dir / "m_tournament_training_dataset_advanced.csv")
w_train_df = pd.read_csv(processed_dir / "w_tournament_training_dataset_advanced.csv")

In [93]:
m_fill_values = m_train_df.reindex(columns=m_features).median(numeric_only=True).to_dict()
w_fill_values = w_train_df.reindex(columns=w_features).median(numeric_only=True).to_dict()

In [94]:
for feature_dict in [m_fill_values, w_fill_values]:
    for k, v in feature_dict.items():
        if pd.isna(v):
            feature_dict[k] = 0.0

if "SeedNumDiff" in m_fill_values:
    m_fill_values["SeedNumDiff"] = 0.0

if "SeedNumDiff" in w_fill_values:
    w_fill_values["SeedNumDiff"] = 0.0

### Load supporting team data

In [95]:
m_team_features = pd.read_csv(processed_dir / "m_team_season_features.csv")
w_team_features = pd.read_csv(processed_dir / "w_team_season_features.csv")

m_teams = set(pd.read_csv(data_dir / "MTeams.csv")["TeamID"].tolist())
w_teams = set(pd.read_csv(data_dir / "WTeams.csv")["TeamID"].tolist())

In [96]:
print("Women's ranking-related columns:")
print([c for c in w_team_features.columns if "rank" in c.lower() or "rating" in c.lower()])


Women's ranking-related columns:
['AvgNetRating', 'WMasseyRating']


### Parse submission IDs and assign gender

In [97]:
submission_df = add_gender(sample_submission, m_teams, w_teams)
submission_season = int(submission_df["Season"].max())

In [98]:
print(f"Latest submission season: {submission_season}")

Latest submission season: 2026


### Split submission rows into men's, women's, and unknown

In [99]:
m_submit = submission_df[submission_df["Gender"] == "M"].copy()
w_submit = submission_df[submission_df["Gender"] == "W"].copy()
unknown_submit = submission_df[submission_df["Gender"] == "UNKNOWN"].copy()

In [100]:
print("Men rows:", len(m_submit))
print("Women rows:", len(w_submit))
print("Unknown rows:", len(unknown_submit))

Men rows: 66430
Women rows: 65703
Unknown rows: 0


### Merge men's team features and engineer submission features

In [101]:
m_submit = merge_team_features(m_submit, m_team_features)
m_submit = add_submission_features(m_submit, "MasseyOrdinalRank")

In [102]:
print("\nMen merged columns check:")
print([c for c in m_submit.columns if "Seed" in c][:20])


Men merged columns check:
['Team1_Seed', 'Team1_SeedNum', 'Team1_HasTournamentSeed', 'Team2_Seed', 'Team2_SeedNum', 'Team2_HasTournamentSeed', 'SeedNumDiff']


### Merge women's team features and engineer submission features

In [103]:
w_submit = merge_team_features(w_submit, w_team_features)
w_submit = add_submission_features(w_submit, "WMasseyRating")

In [104]:
print("\nWomen merged columns check:")
print([c for c in w_submit.columns if "Seed" in c][:20])
print([c for c in w_submit.columns if "WMassey" in c][:20])


Women merged columns check:
['Team1_Seed', 'Team1_SeedNum', 'Team1_HasTournamentSeed', 'Team2_Seed', 'Team2_SeedNum', 'Team2_HasTournamentSeed', 'SeedNumDiff']
['Team1_WMasseyRating', 'Team1_WMasseySOS', 'Team2_WMasseyRating', 'Team2_WMasseySOS']


### Check missing values in required model features

In [105]:
m_submit = ensure_model_features(
    m_submit,
    m_features,
    m_fill_values,
    dataset_name="Men submission"
)

In [106]:
w_submit = ensure_model_features(
    w_submit,
    w_features,
    w_fill_values,
    dataset_name="Women submission"
)

In [107]:
print_missing_summary(m_submit, m_features, "Men")
print_missing_summary(w_submit, w_features, "Women")


Men missing values:
None

Women missing values:
None


### Generate predictions

In [108]:
m_submit["Pred"] = m_model.predict_proba(m_submit[m_features])[:, 1]

In [109]:
w_submit["Pred"] = w_model.predict_proba(w_submit[w_features])[:, 1]

In [110]:
if len(unknown_submit) > 0:
    unknown_submit["Pred"] = 0.5

### Combine final submission

In [111]:
final_submission = pd.concat(
    [
        m_submit[["ID", "Pred"]],
        w_submit[["ID", "Pred"]],
        unknown_submit[["ID", "Pred"]],
    ],
    ignore_index=True
)

In [112]:
final_submission = pd.concat(
    [
        m_submit[["ID", "Pred"]],
        w_submit[["ID", "Pred"]],
        unknown_submit[["ID", "Pred"]],
    ],
    ignore_index=True
)

In [113]:
final_submission = final_submission.merge(
    sample_submission[["ID"]],
    on="ID",
    how="right"
)

In [114]:
final_submission["Pred"] = final_submission["Pred"].fillna(0.5).clip(0.025, 0.975)

### Save submission file

In [115]:
output_path = output_dir / f"submission_{selected_stage}_best_models_combined.csv"
final_submission.to_csv(output_path, index=False)

### Final checks

In [116]:
print(f"Saved submission to: {output_path}")

Saved submission to: ..\submissions\submission_stage2_best_models_combined.csv


In [117]:
print(final_submission.head())

               ID      Pred
0  2026_1101_1102  0.735679
1  2026_1101_1103  0.204695
2  2026_1101_1104  0.220120
3  2026_1101_1105  0.697724
4  2026_1101_1106  0.632385


In [118]:
print(final_submission.shape)

(132133, 2)


In [119]:
print(final_submission.shape)

(132133, 2)
